In [5]:
import pandas as pd
import numpy as np
import pickle
import requests
import datetime
from xgboost import XGBRegressor
from pathlib import Path

OUT = Path("outputs")

# --- 1. LOAD 2025 DATA ---
print("1. Loading 2025 Data...")
clean_2025 = pd.read_pickle(OUT / "clean.pkl").reset_index()
clean_2025["dt"] = pd.to_datetime(clean_2025["dt"])
clean_2025 = clean_2025[["dt", "location", "aqi_index"]]

# --- 2. LOAD 2026 XLSX DATA ---
print("2. Loading 2026 XLSX Data...")
csv_map = {
    "anand_vihar.xlsx": "Anand Vihar",
    "dwarka.xlsx": "Dwarka",
    "igi_airport.xlsx": "IGI Airport",
    "okhla.xlsx": "Okhla Phase III",
    "rohini.xlsx": "Rohini"
}
all_2026_dfs = []

for file, station_name in csv_map.items():
    if not Path(file).exists(): 
        print(f"⚠️ Missing {file}, skipping...")
        continue
    
    df = pd.read_excel(file, header=None)
    header_row = 0
    for i in range(min(15, len(df))):
        row_str = str(df.iloc[i].values).lower()
        if 'date' in row_str and ('pm2.5' in row_str or 'pm25' in row_str):
            header_row = i
            break
            
    df = pd.read_excel(file, header=header_row)
    df.columns = [str(c).lower().strip() for c in df.columns]
    time_cols = [c for c in df.columns if 'date' in c or 'time' in c]
    pm25_cols = [c for c in df.columns if 'pm2.5' in c or 'pm25' in c]
    
    if not time_cols or not pm25_cols: 
        continue
    
    df = df[[time_cols[0], pm25_cols[0]]]
    df.columns = ["dt", "pm2_5"]
    df["dt"] = pd.to_datetime(df["dt"], errors='coerce')
    df["pm2_5"] = pd.to_numeric(df["pm2_5"], errors='coerce')
    df = df.dropna().set_index("dt").resample("1h").mean().reset_index()
    
    def pm_to_aqi(pm):
        if pm <= 30: return pm * 50 / 30
        elif pm <= 60: return 50 + (pm - 30) * 50 / 30
        elif pm <= 90: return 100 + (pm - 60) * 100 / 30
        elif pm <= 120: return 200 + (pm - 90) * 100 / 30
        elif pm <= 250: return 300 + (pm - 120) * 100 / 130
        else: return 400 + (pm - 250) * 100 / 130
        
    df["aqi_index"] = df["pm2_5"].apply(pm_to_aqi)
    df["location"] = station_name
    all_2026_dfs.append(df[["dt", "location", "aqi_index"]])
    print(f"✅ Processed {station_name}")

# Combine 2025 and 2026 Data
final_df = pd.concat([clean_2025] + all_2026_dfs).sort_values(["location", "dt"]).reset_index(drop=True)

# --- 3. DOWNLOAD HISTORICAL WEATHER ---
print("3. Downloading Historical Weather...")
yesterday = (datetime.date.today() - datetime.timedelta(days=1)).strftime("%Y-%m-%d")

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 28.6139, 
    "longitude": 77.2090, 
    "start_date": "2025-01-01", 
    "end_date": yesterday, 
    "hourly": ["temperature_2m", "wind_speed_10m"], 
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params)
if response.status_code != 200 or "hourly" not in response.json():
    print(f"❌ API Error: {response.text}")
else:
    weather_df = pd.DataFrame(response.json()["hourly"])
    weather_df["dt"] = pd.to_datetime(weather_df["time"])
    weather_df = weather_df.set_index("dt").drop(columns=["time"]).rename(columns={"temperature_2m": "temp", "wind_speed_10m": "wind"})
    print(f"✅ Weather downloaded: {len(weather_df)} rows.")

# --- 4. MERGE & BUILD FEATURES ---
print("4. Merging Weather & Building Future Features...")
final_df = pd.merge(final_df, weather_df, left_on="dt", right_index=True, how="left")

def build_features(df, horizon):
    parts = []
    for loc, g in df.groupby("location"):
        g = g.copy().set_index("dt")
        
        for L in (1, 3, 6, 12, 24): 
            g[f"aqi_lag{L}"] = g.aqi_index.shift(L)
            
        # Future Weather Features (The Real Fix)
        g[f"future_wind_{horizon}h"] = g["wind"].shift(-horizon)
        g[f"future_temp_{horizon}h"] = g["temp"].shift(-horizon)
        
        g["target"] = g.aqi_index.shift(-horizon)
        g = g.dropna()
        parts.append(g)
    return pd.concat(parts)

# --- 5. TRAIN MODELS ---
print("5. Training Models...")
for h, name in [(1, "best_model_1h_final.pkl"), (6, "best_model_6h_final.pkl"), (24, "best_model_24h_final.pkl")]:
    print(f"   -> Training {h}h model...")
    feat = build_features(final_df, horizon=h).reset_index()
    
    # FIXED: "location" is NOT in drop_cols anymore. 
    # We want to keep it so pd.get_dummies can encode it.
    drop_cols = ["dt", "target", "aqi_index", "wind", "temp"]
    
    X = pd.get_dummies(feat.drop(columns=[c for c in drop_cols if c in feat.columns]), columns=["location"]).select_dtypes(include=[np.number])
    y = feat["target"]
    
    m = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=42,objective='reg:quantileerror', quantile_alpha=0.9)
    m.fit(X, y)
    
    with open(OUT / name, "wb") as f:
        pickle.dump({"model": m, "columns": list(X.columns)}, f)
    print(f"   ✅ Saved {name}")

print("\n🎉 All FINAL models trained successfully with Future Weather Features!")

1. Loading 2025 Data...
2. Loading 2026 XLSX Data...


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ Processed Anand Vihar
✅ Processed Dwarka


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ Processed IGI Airport


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ Processed Okhla Phase III


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ Processed Rohini
3. Downloading Historical Weather...
✅ Weather downloaded: 16344 rows.
4. Merging Weather & Building Future Features...
5. Training Models...
   -> Training 1h model...
   ✅ Saved best_model_1h_final.pkl
   -> Training 6h model...
   ✅ Saved best_model_6h_final.pkl
   -> Training 24h model...
   ✅ Saved best_model_24h_final.pkl

🎉 All FINAL models trained successfully with Future Weather Features!
